In [2]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import pickle
import networkx as nx
from tqdm.auto import tqdm
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

from rdkit import Chem, DataStructs
from rdkit.Chem import PandasTools
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}

# sys.path.append('/home/mattia/geass')
# from new_geass import Geass
# import utils
# sys.path.append('/home/mattia/aiddtools')
# import fingerprint_tools as fpt

clear_output()

To calculate graph similarity:
$ python node_similarity.py
$ python sgo_similarity.py
$ python clustering.py

## Silhouette Analysis

In [ ]:
import random
import pickle

import numpy as np
import pandas as pd
import networkx as nx
import tmap as tm
from faerun import Faerun

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score


# --------------------------------------------------
# 1) Load similarity matrix and build distance matrix
# --------------------------------------------------
with open("NPA_graph_similarity_atomp.pkl", "rb") as f:
    sim_matrix = pickle.load(f)
keep_ids = df_fragments["NPA_ID"].dropna().unique()
keep_ids = [npa_id for npa_id in keep_ids if npa_id in sim_matrix.index]
sim_matrix = sim_matrix.loc[keep_ids, keep_ids]

S = graph_sim_filtered.to_numpy(dtype=float)
S = np.nan_to_num(S, nan=0.0)
S = np.clip((S + S.T) / 2.0, 0.0, 1.0)
np.fill_diagonal(S, 1.0)

D = 1.0 - S
D = np.clip((D + D.T) / 2.0, 0.0, 1.0)
np.fill_diagonal(D, 0.0)

Z = linkage(squareform(D, checks=False), method="average")

# --------------------------------------------------
# 2) Silhouette analysis
# --------------------------------------------------
thresholds = np.linspace(0.05, 0.95, 37)
rows = []

for t_cut in thresholds:
    labels = fcluster(Z, t=t_cut, criterion="distance")

    label_counts = pd.Series(labels).value_counts()
    n_clusters = int(label_counts.shape[0])
    n_singletons = int((label_counts == 1).sum())
    singleton_fraction = float((label_counts == 1).mean())
    clustered_fraction = float(label_counts[label_counts > 1].sum() / len(labels))

    valid = pd.Series(labels).map(label_counts).to_numpy() > 1

    score_all = np.nan
    score_non_singletons = np.nan

    if 2 <= n_clusters < len(labels):
        try:
            score_all = silhouette_score(D, labels, metric="precomputed")
        except Exception:
            pass

    labels_valid = labels[valid]
    D_valid = D[np.ix_(valid, valid)]

    if len(np.unique(labels_valid)) >= 2 and len(labels_valid) > len(np.unique(labels_valid)):
        try:
            score_non_singletons = silhouette_score(D_valid, labels_valid, metric="precomputed")
        except Exception:
            pass

    rows.append({
        "threshold": t_cut,
        "n_clusters": n_clusters,
        "n_singletons": n_singletons,
        "singleton_fraction": singleton_fraction,
        "clustered_fraction": clustered_fraction,
        "silhouette_all": score_all,
        "silhouette_non_singletons": score_non_singletons
    })

results = pd.DataFrame(rows)
print(results.sort_values("silhouette_non_singletons", ascending=False).head(10).to_string(index=False))